In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import load_config, find_raw_files, convert_xdf_to_fif

cfg = load_config('../configs/eye_eeg_simul.yaml')
xdf_files = find_raw_files(cfg, extension='xdf')
print(f"Found {len(xdf_files)} XDF files")

In [8]:
# Pick the first subject and convert it
test_xdf = xdf_files[0]
print(f"Test conversion with: {test_xdf.name}\n")

success = convert_xdf_to_fif(cfg, test_xdf, overwrite=False, verbose=True)
print(f"\nResult: {'OK' if success else 'skipped or failed'}")

Test conversion with: subj10.xdf

[subj10] already converted — skipping (use overwrite=True to redo)

Result: skipped or failed


In [9]:
from eeg_toolkit import get_subject_dir
import os

subj_dir = get_subject_dir(cfg, 'subj10')  # ajusta si tu primer sujeto es otro
print(f"Subject folder: {subj_dir}\n")
print("Contents:")
for f in sorted(subj_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}  ({size_mb:.1f} MB)")

Subject folder: C:\Users\juapa\OneDrive\Documentos\Proyecto doctorado\Experimentos_proyecto_tesis\Experimentos Eye Tracker\analisis\analisis_eeg\subj10

Contents:
  subj10_event_mapping.txt  (0.0 MB)
  subj10_events_eve.fif  (0.0 MB)
  subj10_raw.fif  (124.3 MB)


In [10]:
import mne
from eeg_toolkit import get_subject_path

raw = mne.io.read_raw_fif(get_subject_path(cfg, 'subj10', 'raw'), preload=False, verbose='WARNING')
events = mne.read_events(get_subject_path(cfg, 'subj10', 'events'))

print(f"Raw: {raw}")
print(f"\nFirst 5 channels: {raw.ch_names[:5]}")
print(f"Sample rate: {raw.info['sfreq']} Hz")
print(f"\nNumber of events: {len(events)}")
print(f"First 5 events:\n{events[:5]}")

import numpy as np
unique_codes, counts = np.unique(events[:, 2], return_counts=True)
print(f"\nUnique event codes ({len(unique_codes)}):")
for code, n in zip(unique_codes, counts):
    print(f"  {code}: {n}")

Raw: <Raw | subj10_raw.fif, 32 x 970757 (1941.5 s), ~31 KiB, data not loaded>

First 5 channels: ['Fp1', 'Fz', 'F3', 'F7', 'FT9']
Sample rate: 500.0 Hz

Number of events: 1680
First 5 events:
[[74851     0    50]
 [75145     0    20]
 [75399     0    51]
 [76154     0    34]
 [76406     0    52]]

Unique event codes (31):
  12: 8
  13: 8
  14: 7
  20: 240
  21: 8
  23: 15
  24: 14
  31: 9
  32: 9
  34: 10
  40: 240
  41: 11
  42: 14
  43: 7
  50: 240
  51: 240
  52: 240
  90: 219
  91: 21
  112: 11
  113: 10
  114: 12
  121: 9
  123: 10
  124: 8
  131: 8
  132: 9
  134: 13
  141: 8
  142: 9
  143: 13


In [11]:
# Verify units are in the expected EEG range
data_chunk = raw.get_data(start=0, stop=2500)  # first 5 seconds (5s × 500 Hz)
print(f"Data range (first 5s):")
print(f"  min = {data_chunk.min():.3e} V  =  {data_chunk.min()*1e6:.0f} µV")
print(f"  max = {data_chunk.max():.3e} V  =  {data_chunk.max()*1e6:.0f} µV")
print(f"  std = {data_chunk.std():.3e} V  =  {data_chunk.std()*1e6:.1f} µV")

Data range (first 5s):
  min = -2.992e-02 V  =  -29923 µV
  max = 6.876e-03 V  =  6876 µV
  std = 1.068e-02 V  =  10681.4 µV


In [12]:
# Sample data from the middle of the recording (more stable, away from start)
data_middle = raw.get_data(start=250000, stop=252500)  # 5 seconds in the middle
print(f"Middle 5s of recording:")
print(f"  min = {data_middle.min()*1e6:.0f} µV")
print(f"  max = {data_middle.max()*1e6:.0f} µV")
print(f"  std = {data_middle.std()*1e6:.1f} µV")

# Per-channel stats
print(f"\nPer-channel std (µV):")
for ch_name, ch_std in zip(raw.ch_names, data_middle.std(axis=1) * 1e6):
    print(f"  {ch_name}: {ch_std:.1f}")

# Apply a quick high-pass filter (1 Hz) to remove DC drift
raw_filtered = raw.copy().load_data().filter(l_freq=1.0, h_freq=None, verbose='WARNING')

data_filt = raw_filtered.get_data(start=250000, stop=252500)
print(f"After 1 Hz high-pass filter (middle 5s):")
print(f"  min = {data_filt.min()*1e6:.1f} µV")
print(f"  max = {data_filt.max()*1e6:.1f} µV")
print(f"  std = {data_filt.std()*1e6:.1f} µV")

Middle 5s of recording:
  min = -29510 µV
  max = 10491 µV
  std = 11166.2 µV

Per-channel std (µV):
  Fp1: 30.1
  Fz: 7.3
  F3: 7.9
  F7: 19.5
  FT9: 19.8
  FC5: 10.4
  FC1: 8.8
  C3: 6.8
  T7: 18.8
  TP9: 20.3
  CP5: 8.8
  CP1: 6.2
  Pz: 9.7
  P3: 10.2
  P7: 9.0
  O1: 8.8
  Oz: 13.6
  O2: 15.0
  P4: 7.2
  P8: 20.3
  TP10: 22.2
  CP6: 10.7
  CP2: 5.7
  Cz: 5.3
  C4: 10.2
  T8: 12.8
  FT10: 11.8
  FC6: 8.1
  FC2: 3.2
  F4: 7.4
  F8: 19.8
  Fp2: 26.7
Reading 0 ... 970756  =      0.000 ...  1941.512 secs...
After 1 Hz high-pass filter (middle 5s):
  min = -106.5 µV
  max = 129.3 µV
  std = 8.3 µV


In [16]:
from eeg_toolkit import load_config, convert_all_xdfs

cfg = load_config('../configs/eye_eeg_simul.yaml')

# Convert all XDFs (subj10 will be skipped because it's already done).
summary = convert_all_xdfs(cfg, overwrite=True, verbose=True)

Found 38 XDF files in C:\Users\juapa\OneDrive\Documentos\Proyecto doctorado\Experimentos_proyecto_tesis\Experimentos Eye Tracker\Datos

--- [1/38] subj10 ---
[subj10] reading subj10.xdf
[subj10]   EEG: 32 ch, 500.0 Hz, 1941.5 s
[subj10]   markers: 1680 events, 31 unique codes
[subj10]   saved: subj10_raw.fif
[subj10]   saved: subj10_events_eve.fif, subj10_event_mapping.txt

--- [2/38] subj13 ---
[subj13] reading subj13.xdf
[subj13]   EEG: 32 ch, 500.0 Hz, 2193.5 s
[subj13]   markers: 1680 events, 31 unique codes
[subj13]   saved: subj13_raw.fif
[subj13]   saved: subj13_events_eve.fif, subj13_event_mapping.txt

--- [3/38] subj14 ---
[subj14] reading subj14.xdf
[subj14]   EEG: 32 ch, 500.0 Hz, 2190.0 s
[subj14]   markers: 1680 events, 32 unique codes
[subj14]   saved: subj14_raw.fif
[subj14]   saved: subj14_events_eve.fif, subj14_event_mapping.txt

--- [4/38] subj15 ---
[subj15] reading subj15.xdf
[subj15]   EEG: 32 ch, 500.0 Hz, 1944.9 s
[subj15]   markers: 1680 events, 32 unique codes


In [17]:
from eeg_toolkit import find_subjects, find_raw_files

# Total folders converted
all_folders = find_subjects(cfg, apply_exclusions=False)
print(f"All converted folders ({len(all_folders)}): {all_folders}")
print()

# After exclusions
included = find_subjects(cfg)
print(f"Subjects to analyze ({len(included)}): {included}")
print()

# Compare with XDFs on disk
xdf_subjects = sorted([f.stem for f in find_raw_files(cfg, extension='xdf')])
missing = set(xdf_subjects) - set(all_folders)
print(f"XDFs without a folder ({len(missing)}): {sorted(missing) if missing else 'none'}")

All converted folders (38): ['subj2', 'subj3', 'subj4', 'subj5', 'subj6', 'subj7', 'subj8', 'subj10', 'subj13', 'subj14', 'subj15', 'subj16', 'subj17', 'subj18', 'subj19', 'subj20', 'subj21', 'subj22', 'subj23', 'subj24', 'subj25', 'subj26', 'subj27', 'subj28', 'subj29', 'subj30', 'subj31', 'subj32', 'subj33', 'subj34', 'subj35', 'subj36', 'subj37', 'subj38', 'subj39', 'subj40', 'subj41', 'subj42']

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj2', 'subj20', 'subj28', 'subj42', 'subj7']
Subjects to analyze (30): ['subj3', 'subj4', 'subj5', 'subj6', 'subj8', 'subj10', 'subj14', 'subj17', 'subj18', 'subj19', 'subj21', 'subj22', 'subj23', 'subj24', 'subj25', 'subj26', 'subj27', 'subj29', 'subj30', 'subj31', 'subj32', 'subj33', 'subj34', 'subj35', 'subj36', 'subj37', 'subj38', 'subj39', 'subj40', 'subj41']

XDFs without a folder (0): none
